In [1]:
"""
台股PSO-BIGRU預測系統
包含：PSO參數優化
不包含：HFSLS特徵選擇（使用全部特徵）
用於消融實驗：驗證PSO優化的效果
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 20
loss = "mse"
measure = [[]]
true_pre = [[], []]
best_result_buffer = None

# ============================================================
# 進度追蹤器
# ============================================================
class ProgressTracker:
    def __init__(self, total_stocks):
        self.total_stocks = total_stocks
        self.start_time = time.time()
        self.stock_start_time = None
        
    def start_stock(self, stock_name, stock_idx):
        self.stock_start_time = time.time()
        print(f"\n{'='*70}")
        print(f"📈 [{stock_idx}/{self.total_stocks}] 處理：{stock_name}")
        print(f"{'='*70}")
    
    def pso_progress(self, iteration, total, best_mse):
        if iteration % 5 == 0 or iteration == total:
            percent = (iteration / total) * 100
            filled = int(30 * iteration / total)
            bar = '█' * filled + '░' * (30 - filled)
            print(f"\r  PSO: |{bar}| {percent:>5.1f}% | MSE: {best_mse:.6f}", end='')
    
    def end_stock(self, stock_name):
        stock_time = time.time() - self.stock_start_time
        print(f"\n{'='*70}")
        print(f"✅ {stock_name} 完成！耗時 {stock_time/60:.1f} 分鐘")
        print(f"{'='*70}")

tracker = None

# ============================================================
# 核心函數
# ============================================================

def process_data(X):
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window:
            x.append(list(que))
    return np.array(x)

def build_bigru_model(params, input_shape):
    """使用PSO優化的參數構建模型"""
    units, dropout_rate, _ = params
    model = Sequential([
        Bidirectional(GRU(int(units), activation='relu', return_sequences=False), 
                     input_shape=input_shape),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def fitness_function(params, x_train, y_train, x_val, y_val):
    try:
        model = build_bigru_model(params, x_train.shape[1:])
        model.fit(x_train, y_train, batch_size=32, epochs=8, verbose=0, shuffle=False)
        y_pred = model.predict(x_val, verbose=0)
        return mean_squared_error(y_val, y_pred)
    except:
        return float('inf')

def PSO(x_train, y_train, x_val, y_val, pop_size=8, max_iter=25):
    """PSO粒子群優化"""
    global tracker
    
    param_bounds = np.array([[32, 96], [0.1, 0.4], [0.0005, 0.005]])
    population = np.random.uniform(param_bounds[:, 0], param_bounds[:, 1], (pop_size, 3))
    velocities = np.random.uniform(-1, 1, (pop_size, 3))
    
    pbest_pos = population.copy()
    pbest_score = np.full(pop_size, float('inf'))
    gbest_pos = np.zeros(3)
    gbest_score = float('inf')
    
    for t in range(max_iter):
        for i in range(pop_size):
            fitness = fitness_function(population[i], x_train, y_train, x_val, y_val)
            
            if fitness < pbest_score[i]:
                pbest_score[i] = fitness
                pbest_pos[i] = population[i]
            
            if fitness < gbest_score:
                gbest_score = fitness
                gbest_pos = population[i]
        
        w, c1, c2 = 0.5, 1.5, 1.5
        for i in range(pop_size):
            r1, r2 = np.random.rand(2)
            velocities[i] = (w * velocities[i] +
                           c1 * r1 * (pbest_pos[i] - population[i]) +
                           c2 * r2 * (gbest_pos - population[i]))
            population[i] = np.clip(population[i] + velocities[i], 
                                   param_bounds[:, 0], param_bounds[:, 1])
        
        if tracker:
            tracker.pso_progress(t+1, max_iter, gbest_score)
    
    print(f"\n  ✓ PSO最佳參數：units={int(gbest_pos[0])}, dropout={gbest_pos[1]:.3f}")
    return gbest_pos

def train_pso_bigru(x_train, y_train, x_test, y_test, test_indices, dates, stock_name):
    """訓練PSO-BIGRU模型"""
    global best_result_buffer
    
    print(f"  🔍 PSO參數優化中...")
    best_params = PSO(x_train, y_train, x_test, y_test)
    
    print(f"  🏗️  構建最終模型...")
    model = Sequential([
        Bidirectional(GRU(int(best_params[0]), activation='relu', return_sequences=False),
                     input_shape=x_train.shape[1:]),
        Dropout(best_params[1]),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss=loss)
    
    print(f"  🔄 開始訓練...")
    model.fit(x_train, y_train, batch_size=32, epochs=30,
             validation_split=0.1, verbose=0, shuffle=False)
    
    # 預測與評估
    y_pred = model.predict(x_test, verbose=0)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"  📊 MSE={mse:.6f}, R²={r2:.4f}")
    
    # 保存結果
    y_pred_inv = scaler.inverse_transform(y_pred)
    y_test_inv = scaler.inverse_transform(y_test)
    
    measure[0] = [mse, rmse, mae, mape, r2]
    true_pre[0] = list(y_test_inv[:, 0])
    true_pre[1] = list(y_pred_inv[:, 0])
    
    base_dates = [dates[idx] for idx in test_indices]
    predict_dates = [d + pd.Timedelta(days=15) for d in base_dates]
    
    best_result_buffer = pd.DataFrame({
        'stock_name': stock_name,
        'base_date': base_dates,
        'predict_date': predict_dates,
        'predicted_close': y_pred_inv[:, 0],
        'actual_close': y_test_inv[:, 0],
        'error': y_pred_inv[:, 0] - y_test_inv[:, 0],
        'error_pct': ((y_pred_inv[:, 0] - y_test_inv[:, 0]) / y_test_inv[:, 0] * 100)
    })
    
    return model

def save_final_result():
    """保存最終結果"""
    global best_result_buffer
    
    if best_result_buffer is None:
        return
    
    output_path = 'D:/pythonProject/a-lunwen/data/predicted_pso_bigru.csv'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    try:
        existing = pd.read_csv(output_path)
        stock_name = best_result_buffer['stock_name'].iloc[0]
        existing = existing[existing['stock_name'] != stock_name]
        existing = pd.concat([existing, best_result_buffer], ignore_index=True)
    except FileNotFoundError:
        existing = best_result_buffer
    
    existing.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    stock_name = best_result_buffer['stock_name'].iloc[0]
    result_count = len(best_result_buffer)
    print(f"\n  💾 已保存 {stock_name} 的結果（{result_count}筆預測）")

def fit_pso_bigru(df, dates, stock_name):
    """PSO-BIGRU訓練（無特徵選擇）"""
    print(f"\n📋 數據：{df.shape[0]}樣本 × {df.shape[1]}特徵")
    print(f"  ⚠️  使用全部特徵（無HFSLS特徵選擇）")
    print(f"  ⚠️  使用PSO參數優化")
    
    X = df.iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    
    y = df['Close'].values[:-window+1]
    y = scaler.fit_transform(y.reshape(-1, 1))
    valid_dates = dates[:-window+1].reset_index(drop=True)
    
    x = process_data(X.values)
    
    split_point = int(len(y) * 0.7)
    x_train = x[:split_point]
    y_train = y[:split_point]
    x_test = x[split_point:]
    y_test = y[split_point:]
    test_indices = list(range(split_point, len(y)))
    
    print(f"  📊 訓練集：{len(x_train)} 樣本")
    print(f"  📊 測試集：{len(x_test)} 樣本")
    
    model = train_pso_bigru(x_train, y_train, x_test, y_test,
                           test_indices, valid_dates, stock_name)
    
    return model

def main(data, stock_name):
    """主函數"""
    global measure, true_pre, best_result_buffer
    
    measure = [[]]
    true_pre = [[], []]
    best_result_buffer = None
    
    df_full = pd.read_csv(f'D:/pythonProject/a-lunwen/data/{data}')
    
    if 'Date' in df_full.columns:
        dates = pd.to_datetime(df_full['Date'])
        df = df_full.drop(columns=['Date'])
    else:
        dates = pd.date_range('2021-01-01', periods=len(df_full), freq='D')
        df = df_full
    
    df = df.fillna(0).sort_index()
    
    fit_pso_bigru(df, dates, stock_name)
    save_final_result()
    
    print(f"\n{'='*70}")
    print(f"🎉 {stock_name} 完成！")
    print(f"{'='*70}")
    print(f"📈 最終指標：")
    print(f"   MSE:  {measure[0][0]:.6f}")
    print(f"   RMSE: {measure[0][1]:.6f}")
    print(f"   MAE:  {measure[0][2]:.6f}")
    print(f"   MAPE: {measure[0][3]:.4f}")
    print(f"   R²:   {measure[0][4]:.4f}")
    print(f"{'='*70}\n")

# ============================================================
# 主程序
# ============================================================
if __name__ == '__main__':
    print("="*70)
    print("🚀 台股PSO-BIGRU預測系統")
    print("   包含：PSO參數優化")
    print("   不含：HFSLS特徵選擇（使用全部特徵）")
    print("   預測：11個交易日後的收盤價")
    print("="*70)
    print("📌 用於消融實驗：驗證PSO優化的效果")
    print("="*70)
    
    stock_list = ['台積電', '長榮', '聯發科', '統一超']
    tracker = ProgressTracker(len(stock_list))
    
    total_start = time.time()
    success = 0
    all_results = []
    
    for idx, stock in enumerate(stock_list, 1):
        try:
            tracker.start_stock(stock, idx)
            main(f"{stock}_date.csv", stock)
            tracker.end_stock(stock)
            
            all_results.append({
                'stock': stock,
                'MSE': measure[0][0],
                'RMSE': measure[0][1],
                'MAE': measure[0][2],
                'MAPE': measure[0][3],
                'R²': measure[0][4]
            })
            
            success += 1
            
        except Exception as e:
            print(f"\n❌ {stock} 失敗：{e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*70}")
    print(f"🏁 PSO-BIGRU訓練完成！")
    print(f"{'='*70}")
    print(f"✅ 成功：{success}/{len(stock_list)}")
    print(f"⏱️  總耗時：{(time.time()-total_start)/60:.1f} 分鐘")
    print(f"{'='*70}")
    
    if all_results:
        print(f"\n📊 PSO-BIGRU性能總結")
        print(f"{'='*70}")
        print(f"{'股票':<10} {'MSE':<12} {'RMSE':<12} {'R²':<10}")
        print(f"{'-'*70}")
        
        avg_mse = avg_rmse = avg_r2 = 0
        
        for result in all_results:
            print(f"{result['stock']:<10} {result['MSE']:<12.6f} {result['RMSE']:<12.6f} {result['R²']:<10.4f}")
            avg_mse += result['MSE']
            avg_rmse += result['RMSE']
            avg_r2 += result['R²']
        
        print(f"{'-'*70}")
        print(f"{'平均':<10} {avg_mse/len(all_results):<12.6f} {avg_rmse/len(all_results):<12.6f} {avg_r2/len(all_results):<10.4f}")
        print(f"{'='*70}")
    
    print(f"\n💾 結果：D:/pythonProject/a-lunwen/data/predicted_pso_bigru.csv")
    print(f"{'='*70}")

🚀 台股PSO-BIGRU預測系統
   包含：PSO參數優化
   不含：HFSLS特徵選擇（使用全部特徵）
   預測：11個交易日後的收盤價
📌 用於消融實驗：驗證PSO優化的效果

📈 [1/4] 處理：台積電

📋 數據：1141樣本 × 159特徵
  ⚠️  使用全部特徵（無HFSLS特徵選擇）
  ⚠️  使用PSO參數優化
  📊 訓練集：785 樣本
  📊 測試集：337 樣本
  🔍 PSO參數優化中...
  PSO: |██████████████████████████████| 100.0% | MSE: 0.005365
  ✓ PSO最佳參數：units=51, dropout=0.100
  🏗️  構建最終模型...
  🔄 開始訓練...
  📊 MSE=0.074062, R²=-3.8926

  💾 已保存 台積電 的結果（337筆預測）

🎉 台積電 完成！
📈 最終指標：
   MSE:  0.074062
   RMSE: 0.272143
   MAE:  0.263009
   MAPE: 0.3433
   R²:   -3.8926


✅ 台積電 完成！耗時 42.4 分鐘

📈 [2/4] 處理：長榮

📋 數據：1141樣本 × 159特徵
  ⚠️  使用全部特徵（無HFSLS特徵選擇）
  ⚠️  使用PSO參數優化
  📊 訓練集：785 樣本
  📊 測試集：337 樣本
  🔍 PSO參數優化中...
  PSO: |██████████████████████████████| 100.0% | MSE: 0.001647
  ✓ PSO最佳參數：units=81, dropout=0.266
  🏗️  構建最終模型...
  🔄 開始訓練...
  📊 MSE=0.001534, R²=0.8146

  💾 已保存 長榮 的結果（337筆預測）

🎉 長榮 完成！
📈 最終指標：
   MSE:  0.001534
   RMSE: 0.039161
   MAE:  0.030562
   MAPE: 0.0412
   R²:   0.8146


✅ 長榮 完成！耗時 50.1 分鐘

📈 [3/4] 處理：聯發科

📋 數據：1141樣本 × 159特徵
  ⚠️  使用全